# 读取 parquet 文件格式

In [4]:
import sys
print("Python 路径:", sys.executable)
import pyarrow
import pyarrow.parquet as pq
import pandas as pd
print("pyarrow :", pyarrow.__version__)
print("pandas:", pd.__version__)
import numpy as np
import time

Python 路径: /public/home/zhaoshuo/miniconda3/envs/casmi_env/bin/python
pyarrow : 25.0.1
pandas: 2.3.3


In [3]:
def inspect(path: str, n_rows: int = 3):
    """打印 Parquet 文件的元信息、schema 和前 n 行。"""
    print("=" * 60)
    print(f"文件: {path}")
    print("=" * 60)

    # ---- 元信息 ----
    meta = pq.read_metadata(path)
    print(f"\n【元信息】")
    print(f"  总行数   : {meta.num_rows:,}")
    print(f"  总列数   : {meta.num_columns}")
    print(f"  行组数   : {meta.num_row_groups}")

    # ---- schema（Arrow 类型）----
    pf = pq.ParquetFile(path)
    print(f"\n【Schema（列名 + 类型）】")
    print(pf.schema_arrow)

    # ---- 前 n 行 ----
    print(f"\n【前 {n_rows} 行】")
    batch = next(pf.iter_batches(batch_size=n_rows))
    df = batch.to_pandas()
    print(df.to_string())

    # ---- 每列的统计 ----
    print(f"\n【每列类型 + 非空数量】")
    for col in df.columns:
        non_null = df[col].notna().sum()
        print(f"  {col:20s}  type={str(df[col].dtype):15s}  "
              f"非空={non_null}/{len(df)}")

In [4]:
pwd

'/public/home/zhaoshuo/casmi/notebooks'

In [7]:
# 查看 test.parquet 数据集
inspect("../data/enveda_casmi26/test.parquet")

文件: ../data/enveda_casmi26/test.parquet

【元信息】
  总行数   : 1,213
  总列数   : 12
  行组数   : 1

【Schema（列名 + 类型）】
molecule_id: large_string
spectrum_id: large_string
ms2_mzs: list<element: double>
  child 0, element: double
ms2_normalized_intensities: list<element: double>
  child 0, element: double
base_peak_intensity: double
adduct: large_string
ionization_mode: large_string
instrument_type: large_string
precursor_mz: double
collision_energy_orig: large_string
collision_energy_ev: list<element: double>
  child 0, element: double
collision_energy_orig_units: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1663

【前 3 行】
  molecule_id spectrum_id                                                                                                                                                                                                                                                                                                  

In [8]:
# 查看 train.parquet 数据集
inspect("../data/enveda_casmi26/train.parquet")

文件: ../data/enveda_casmi26/train.parquet

【元信息】
  总行数   : 2,539,608
  总列数   : 18
  行组数   : 21

【Schema（列名 + 类型）】
ingest_lib: string
normalized_smiles: string
inchikey: string
inchikey14: string
molecular_formula: string
ionization_mode: string
instrument_type: string
adduct: string
adduct_orig: string
precursor_mz: double
precursor_error_ppm: double
ms2_mzs: list<element: double>
  child 0, element: double
ms2_normalized_intensities: list<element: double>
  child 0, element: double
num_peaks: int64
base_peak_intensity: double
collision_energy_ev: list<element: double>
  child 0, element: double
collision_energy_orig: string
collision_energy_orig_units: string

【前 3 行】
  ingest_lib                                      normalized_smiles                     inchikey      inchikey14 molecular_formula ionization_mode instrument_type   adduct adduct_orig  precursor_mz  precursor_error_ppm                                                                                                         

In [9]:
# 查看数据来源
pf = pq.ParquetFile("../data/enveda_casmi26/train.parquet")
libs = []
for batch in pf.iter_batches(batch_size=100000, columns=["ingest_lib"]):
    libs.append(batch.to_pandas())
libs_df = pd.concat(libs)
print(libs_df["ingest_lib"].value_counts())

ingest_lib
enveda-180            1153785
pluskal_ms2            527581
riken                  347171
gnps                   220849
massbank               101727
mona                    92416
spectraverse            50933
msdial                  40765
drug_plus                2545
enveda-np-examples       1184
masaryk                   652
Name: count, dtype: int64


In [10]:
# 用 inchikey14 统计唯一分子数
unique_keys = set()
for batch in pf.iter_batches(batch_size=100000, columns=["inchikey14"]):
    unique_keys.update(batch.to_pandas()["inchikey14"].unique())
print("唯一分子数:", len(unique_keys))

唯一分子数: 275810


In [14]:
# 统计训练集字段完整度
pf = pq.ParquetFile("../data/enveda_casmi26/train.parquet")

# 关键列：normalized_smiles, inchikey14, molecular_formula
cols = ["normalized_smiles", "inchikey14", "molecular_formula",
        "precursor_mz", "adduct", "ms2_mzs",
        "instrument_type", "base_peak_intensity"]
total = 0
non_null = {c: 0 for c in cols}

for batch in pf.iter_batches(batch_size=100000, columns=cols):
    df = batch.to_pandas()
    total += len(df)
    for c in cols:
        non_null[c] += df[c].notna().sum()

print(f"总行数: {total:,}")
print("\n各列非空比例:")
for c in cols:
    pct = 100 * non_null[c] / total
    print(f"  {c:30s} {non_null[c]:>10,}  ({pct:.1f}%)")

总行数: 2,539,608

各列非空比例:
  normalized_smiles               2,539,608  (100.0%)
  inchikey14                      2,539,608  (100.0%)
  molecular_formula               2,539,608  (100.0%)
  precursor_mz                    2,539,608  (100.0%)
  adduct                          2,539,608  (100.0%)
  ms2_mzs                         2,539,608  (100.0%)
  instrument_type                 2,506,555  (98.7%)
  base_peak_intensity             1,376,470  (54.2%)


In [13]:
# 统计唯一分子数和平均每分子谱图数
unique_keys = set()
per_key_count = {}

for batch in pf.iter_batches(batch_size=100000, 
                              columns=["inchikey14"]):
    keys = batch.to_pandas()["inchikey14"]
    for k in keys:
        unique_keys.add(k)
        per_key_count[k] = per_key_count.get(k, 0) + 1

import numpy as np
counts = np.array(list(per_key_count.values()))
print(f"唯一分子数: {len(unique_keys):,}")
print(f"每分子谱图数: 平均={counts.mean():.1f}, "
      f"中位={np.median(counts):.0f}, "
      f"最大={counts.max()}")

唯一分子数: 275,810
每分子谱图数: 平均=9.2, 中位=6, 最大=1468


In [8]:
def inspect_mz_range(path, batch_size=100_000):
    pf = pq.ParquetFile(path)

    # ---- 合理范围（基于化学常识 + 比赛描述）----
    PREC_MIN = 50.0        # 前体最小合理值
    PREC_MAX = 2000.0      # 前体最大合理值（比赛 1159，留余量）
    FRAG_MIN = 5.0         # 碎片最小合理值
    FRAG_MAX = 2000.0      # 碎片最大合理值

    # ---- 统计 ----
    n_rows = 0
    n_anomaly_prec = 0
    n_anomaly_frag = 0
    n_empty = 0

    prec_clean = []        # 干净的前体 m/z
    frag_min_clean = float("inf")
    frag_max_clean = float("-inf")

    rng = np.random.default_rng(42)
    frag_samples = []

    t0 = time.time()
    for batch in pf.iter_batches(
        batch_size=batch_size,
        columns=["precursor_mz", "ms2_mzs"]
    ):
        df = batch.to_pandas()
        n_rows += len(df)

        prec = df["precursor_mz"].dropna().values
        # 过滤异常前体
        mask_ok = (prec >= PREC_MIN) & (prec <= PREC_MAX)
        n_anomaly_prec += (~mask_ok).sum()
        prec_clean.append(prec[mask_ok])

        # 碎片
        for mzs in df["ms2_mzs"]:
            if mzs is None or len(mzs) == 0:
                n_empty += 1
                continue
            arr = np.asarray(mzs, dtype=np.float64)
            # 过滤异常碎片
            mask = (arr >= FRAG_MIN) & (arr <= FRAG_MAX)
            n_anomaly_frag += (~mask).sum()
            arr_clean = arr[mask]
            if len(arr_clean) == 0:
                continue
            frag_min_clean = min(frag_min_clean, float(arr_clean.min()))
            frag_max_clean = max(frag_max_clean, float(arr_clean.max()))
            # 采样
            k = min(1000, len(arr_clean))
            idx = rng.choice(len(arr_clean), size=k, replace=False)
            frag_samples.append(arr_clean[idx])

        print(f"  已处理 {n_rows:,} 行  ({time.time()-t0:.0f} 秒)", end="\r")

    print(f"\n  完成: {n_rows:,} 行，用时 {time.time()-t0:.1f} 秒")

    prec_clean = np.concatenate(prec_clean)
    frag_samples = np.concatenate(frag_samples)

    # ---- 异常统计 ----
    print("\n" + "=" * 60)
    print("数据质量")
    print("=" * 60)
    print(f"  总行数:             {n_rows:>12,}")
    print(f"  空谱图:             {n_empty:>12,}")
    print(f"  异常前体 (滤掉):    {n_anomaly_prec:>12,}  "
          f"({100*n_anomaly_prec/n_rows:.3f}%)")
    print(f"  异常碎片 (滤掉):    {n_anomaly_frag:>12,}")

    # ---- 干净 precursor_mz ----
    print("\n" + "=" * 60)
    print(f"干净的 precursor_mz（过滤到 [{PREC_MIN}, {PREC_MAX}]）")
    print("=" * 60)
    print(f"  count : {len(prec_clean):,}")
    print(f"  min   : {prec_clean.min():.2f}")
    print(f"  max   : {prec_clean.max():.2f}")
    for q in [50, 90, 99, 99.9, 99.99]:
        print(f"  {q:>5}%: {np.percentile(prec_clean, q):.2f}")

    # ---- 干净碎片 ----
    print("\n" + "=" * 60)
    print(f"干净的碎片 m/z（过滤到 [{FRAG_MIN}, {FRAG_MAX}]）")
    print("=" * 60)
    print(f"  全局 min : {frag_min_clean:.2f}")
    print(f"  全局 max : {frag_max_clean:.2f}")
    print(f"  采样数   : {len(frag_samples):,}")
    for q in [0.01, 0.1, 1, 50, 99, 99.9, 99.99]:
        print(f"  {q:>6}%: {np.percentile(frag_samples, q):.2f}")

    # ---- 建议 ----
    print("\n" + "=" * 60)
    print("建议 MZ_MIN / MZ_MAX")
    print("=" * 60)
    frag_p9999 = np.percentile(frag_samples, 99.99)
    print(f"  MZ_MIN = {int(np.floor(frag_min_clean))}  "
          f"(最低碎片 {frag_min_clean:.1f})")
    print(f"  MZ_MAX = {int(np.ceil(frag_p9999 / 100) * 100)}  "
          f"(99.99% 分位数 {frag_p9999:.1f})")

In [9]:
inspect_mz_range("../data/enveda_casmi26/train.parquet")

  已处理 2,539,608 行  (67 秒)
  完成: 2,539,608 行，用时 67.5 秒

数据质量
  总行数:                2,539,608
  空谱图:                        0
  异常前体 (滤掉):              77  (0.003%)
  异常碎片 (滤掉):          20,553

干净的 precursor_mz（过滤到 [50.0, 2000.0]）
  count : 2,539,531
  min   : 56.97
  max   : 1986.10
     50%: 350.11
     90%: 699.45
     99%: 975.62
   99.9%: 1382.70
  99.99%: 1833.99

干净的碎片 m/z（过滤到 [5.0, 2000.0]）
  全局 min : 5.00
  全局 max : 2000.00
  采样数   : 350,883,458
    0.01%: 20.08
     0.1%: 20.85
       1%: 28.90
      50%: 161.08
      99%: 685.17
    99.9%: 1453.31
   99.99%: 1917.91

建议 MZ_MIN / MZ_MAX
  MZ_MIN = 5  (最低碎片 5.0)
  MZ_MAX = 2000  (99.99% 分位数 1917.9)


In [10]:
pf = pq.ParquetFile("../data/enveda_casmi26/train.parquet")
anomalies = []

for batch in pf.iter_batches(batch_size=100000,
                              columns=["precursor_mz", "adduct",
                                       "ingest_lib", "ms2_mzs",
                                       "molecular_formula"]):
    df = batch.to_pandas()
    mask = ~df["precursor_mz"].between(50.0, 2000.0)
    if mask.any():
        anomalies.append(df[mask])

anom = pd.concat(anomalies, ignore_index=True)
print(f"异常前体数: {len(anom)}")
print("\n前体 m/z 分布:")
print(anom["precursor_mz"].describe())
print("\n按来源库:")
print(anom["ingest_lib"].value_counts())
print("\n按加合物:")
print(anom["adduct"].value_counts())
print("\n原始前体值:")
print(anom["precursor_mz"].values[:20])

异常前体数: 77

前体 m/z 分布:
count    7.700000e+01
mean     3.228825e+04
std      2.768803e+05
min      2.000000e+00
25%      4.503400e+01
50%      4.600000e+01
75%      2.001835e+03
max      2.430333e+06
Name: precursor_mz, dtype: float64

按来源库:
ingest_lib
gnps         31
riken        16
msdial       11
massbank     10
mona          8
drug_plus     1
Name: count, dtype: int64

按加合物:
adduct
[M+H]+      59
[M+K]+       9
[M-H]-       5
[M+2H]2+     3
[M+Na]+      1
Name: count, dtype: int64

原始前体值:
[  32.0495 2164.96   2186.94   2162.94     34.0293   34.0293   34.0293
   32.05     32.05     32.05   3355.6      45.0444 2164.96     45.034
   45.034    45.034    44.9851   44.9851   44.9851 2001.84  ]
